# CineEmbed — 03 Train Contrastive (Phase 1 sweep)

Self-supervised pretext stage that pulls together two modality-dropout views of the same film and pushes apart different films. After pretext the **backbone** is saved (projection head discarded per Chen et al. 2020) and is meant to be loaded by `02_train_ae.ipynb` / `04_train_dec.ipynb` for fine-tuning.

Spec: `docs/superpowers/specs/2026-05-06-clustering-improvement-techniques.md` §2.1.

**Sweep grid (3 configs, ~30 min each on T4):**

| run_name | tau | drop_prob | proj_dim |
|---|---|---|---|
| `contrastive_tau0p1_drop0p3` | 0.1 | 0.3 | 128 |
| `contrastive_tau0p5_drop0p3` | 0.5 | 0.3 | 128 |
| `contrastive_tau0p1_drop0p4` | 0.1 | 0.4 | 128 |

**Outputs (per run, under `<artifacts>/models/<run_name>/`):**
- `pretext_backbone.pt` — load this into `MultiModalBackbone` for fine-tune
- `pretext_full.pt` — backbone + projection, for resume
- `history.json`, `eval.json`, `umap.png` (when `--umap` flag set)

All training + eval metrics + backbone artifact are pushed to a single W&B run per config under `group=phase-1-sweep`.

Run `00_colab_setup.ipynb` once per session first.

In [2]:
import sys, json
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    try:
        from google.colab import userdata
        _token = userdata.get('GITHUB_TOKEN')
        REPO_URL = f"https://{_token}@github.com/bkaankaya/CineEmbed-A-Multi-Modal-Unsupervised-Film-Recommendation-System.git"
    except Exception:
        REPO_URL = "https://github.com/bkaankaya/CineEmbed-A-Multi-Modal-Unsupervised-Film-Recommendation-System.git"
    REPO_ROOT = Path('/content/cineembed-repo')
    ARTIFACTS = Path('/content/drive/MyDrive/CineEmbed/artifacts')
    if not REPO_ROOT.exists():
        get_ipython().system(f'git clone {REPO_URL} {REPO_ROOT}')
    else:
        get_ipython().system(f'cd {REPO_ROOT} && git fetch -q && git checkout main -q && git pull -q')
    get_ipython().system(f'pip install -e {REPO_ROOT} -q')
    get_ipython().system('pip install wandb -q')
else:
    REPO_ROOT = Path('..').resolve()
    ARTIFACTS = REPO_ROOT / 'artifacts'

sys.path.insert(0, str(REPO_ROOT / 'src'))

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Repo:      {REPO_ROOT}")
print(f"Artifacts: {ARTIFACTS}")
print(f"Device:    {DEVICE}")

# Sanity-check artifacts before launching expensive runs
for p in [ARTIFACTS / 'feature_matrix.npz', ARTIFACTS / 'movies_eda_final.csv']:
    assert p.exists(), f"Missing artifact: {p}"

# Confirm latest script is on disk (must include 5fa95ff or later)
get_ipython().system(f'cd {REPO_ROOT} && git log --oneline -1')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for cineembed (pyproject.toml) ... done
Repo:      /content/cineembed-repo
Artifacts: /content/drive/MyDrive/CineEmbed/artifacts
Device:    cuda
17d6fbb (HEAD -> main, origin/main) fix(03_train_contrastive): skip-if-done guard in sweep loop


## W&B configuration (auto-detected — no interactive prompt)

| Setup                                              | Behavior                                                                    |
| -------------------------------------------------- | --------------------------------------------------------------------------- |
| Colab Secret `WANDB_API_KEY` present               | Online: runs stream to `wandb.ai/<your-entity>/cineembed`.                  |
| No secret + `WANDB_PROJECT = None`                 | Disabled. Only console + history.json / eval.json on Drive.                 |
| No secret + `WANDB_PROJECT` set                    | Offline. Runs land in `wandb/offline-run-*/`. Sync later with `wandb sync`. |

**Set the secret once**: Colab sidebar → **Tools → Secrets → New secret** named `WANDB_API_KEY`. Copy value from <https://wandb.ai/authorize>. Toggle **Notebook access** ON. Then re-run this cell.

This cell never prompts interactively — the prior run's `KeyboardInterrupt` cascade was from `wandb.init` reading an empty `WANDB_API_KEY` env var and falling back to a stdin prompt that Colab subprocesses cannot answer.

In [ ]:
import os

WANDB_PROJECT = 'cineembed'       # set None to skip W&B entirely
WANDB_ENTITY  = None              # e.g. 'team3-seng474' — None = your default from netrc
WANDB_GROUP   = 'phase-1-sweep'   # all 3 runs grouped together on the dashboard

WANDB_MODE = None
WANDB_KEY_PRESENT = False

# Step 1: try Colab Secret first (preferred — never lands in the notebook source).
if WANDB_PROJECT and IN_COLAB:
    try:
        from google.colab import userdata
        _key = userdata.get('WANDB_API_KEY')
        if _key:
            os.environ['WANDB_API_KEY'] = _key
            WANDB_KEY_PRESENT = True
    except Exception:
        pass

# Step 2: fall back to env var (set externally, e.g. `os.environ[...] = ...` in a
# separate scratch cell that is NEVER committed; do not paste keys into this file).
# Works in both Colab and local.
if WANDB_PROJECT and not WANDB_KEY_PRESENT and os.environ.get('WANDB_API_KEY'):
    WANDB_KEY_PRESENT = True

# Step 3: local-only fallback to ~/.netrc set by `wandb login`.
if WANDB_PROJECT and not WANDB_KEY_PRESENT and not IN_COLAB:
    if Path.home().joinpath('.netrc').exists():
        WANDB_KEY_PRESENT = True

if WANDB_PROJECT is None:
    WANDB_MODE = 'disabled'
    print('W&B disabled (WANDB_PROJECT is None).')
elif WANDB_KEY_PRESENT:
    WANDB_MODE = 'online'
    os.environ['WANDB_MODE'] = 'online'
    print(f"W&B ONLINE  project='{WANDB_PROJECT}'  group='{WANDB_GROUP}'")
else:
    # API key absent — go offline rather than killing the subprocess with a
    # stdin prompt that cannot be answered.
    WANDB_MODE = 'offline'
    os.environ['WANDB_MODE'] = 'offline'
    print('No WANDB_API_KEY found → using OFFLINE mode.')
    print('Add WANDB_API_KEY via Tools → Secrets + restart runtime, then re-run.')
    print(f"Offline runs land under wandb/ in the working dir; sync later with `wandb sync`.")

print(f"\nFinal: WANDB_MODE={WANDB_MODE}")

## Sweep — 3 configs
Each config invokes `scripts/train_contrastive.py` in a subprocess so per-run W&B context is isolated. ~25-40 min per run on T4.

Subprocess inherits `WANDB_MODE` + `WANDB_API_KEY` from this notebook's env. Set `INCLUDE_UMAP = True` to render + log UMAP per run (adds ~3-5 min).

In [ ]:
SWEEP = [
    {'run_name': 'contrastive_tau0p1_drop0p3', 'tau': 0.1, 'drop_prob': 0.3, 'proj_dim': 128},
    {'run_name': 'contrastive_tau0p5_drop0p3', 'tau': 0.5, 'drop_prob': 0.3, 'proj_dim': 128},
    {'run_name': 'contrastive_tau0p1_drop0p4', 'tau': 0.1, 'drop_prob': 0.4, 'proj_dim': 128},
]

EPOCHS = 60
BATCH_SIZE = 1024
PATIENCE = 8
INCLUDE_UMAP = False   # True → render + log UMAP per run (adds ~5-15 min)
FORCE = False          # True → re-train even if pretext_backbone.pt + eval.json exist

SCRIPT = REPO_ROOT / 'scripts' / 'train_contrastive.py'
assert SCRIPT.exists(), f"Missing {SCRIPT}"

for cfg in SWEEP:
    run_dir = ARTIFACTS / 'models' / cfg['run_name']
    ckpt = run_dir / 'pretext_backbone.pt'
    evalj = run_dir / 'eval.json'
    print('\n' + '#' * 78)
    print(f"# {cfg['run_name']}  (mode={WANDB_MODE})")
    print('#' * 78)
    if (not FORCE) and ckpt.exists() and evalj.exists():
        print(f"SKIP — already trained. ckpt={ckpt.stat().st_size//1024} KB, eval={evalj.stat().st_size} B.")
        print(f"      Set FORCE=True to overwrite, or delete {run_dir} to redo.")
        continue
    parts = [
        f"python {SCRIPT}",
        f"--artifacts {ARTIFACTS}",
        f"--run-name {cfg['run_name']}",
        f"--tau {cfg['tau']} --drop-prob {cfg['drop_prob']} --proj-dim {cfg['proj_dim']}",
        f"--batch-size {BATCH_SIZE} --epochs {EPOCHS} --patience {PATIENCE}",
        f"--device {DEVICE}",
    ]
    if WANDB_PROJECT:
        parts.append(f"--wandb-project {WANDB_PROJECT}")
        parts.append(f"--wandb-mode {WANDB_MODE}")
        if WANDB_GROUP:
            parts.append(f"--wandb-group {WANDB_GROUP}")
        if WANDB_ENTITY:
            parts.append(f"--wandb-entity {WANDB_ENTITY}")
    if INCLUDE_UMAP:
        parts.append('--umap')
    cmd = ' '.join(parts)
    print('>', cmd)
    rc = get_ipython().system(cmd)
    if rc:
        print(f"!! {cfg['run_name']} returned non-zero: {rc}")

## Summary — collect eval.json from each run

In [ ]:
rows = []
for cfg in SWEEP:
    p = ARTIFACTS / 'models' / cfg['run_name'] / 'eval.json'
    if not p.exists():
        print(f"  (missing eval.json for {cfg['run_name']})")
        continue
    e = json.loads(p.read_text())
    km = e.get('kmeans_k21', {})
    gm = e.get('gmm_k21', {})
    pa = e.get('per_axis_k_kmeans', {})
    ml = e.get('multilabel_genre_macro_nmi_kmeans', {})
    rows.append({
        'run':            cfg['run_name'],
        'tau':            cfg['tau'],
        'drop':           cfg['drop_prob'],
        'km_gNMI':        km.get('genre_nmi'),
        'km_dNMI':        km.get('decade_nmi'),
        'km_lNMI':        km.get('lang_nmi'),
        'gmm_gNMI':       gm.get('genre_nmi'),
        'per_axis_gNMI':  pa.get('genre_nmi_k21'),
        'per_axis_dNMI':  pa.get('decade_nmi_k12'),
        'per_axis_lNMI':  pa.get('lang_nmi_k11'),
        'ml_macro_nmi':   (ml.get('macro_nmi') if isinstance(ml, dict) else None),
    })

try:
    import pandas as pd
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
except ImportError:
    for r in rows:
        print(r)

print('\nMVP baseline (for comparison): dec_z64_k21 genre_NMI = 0.332')

if WANDB_MODE == 'offline':
    print("\nUpload offline runs later with:")
    print(f"  wandb sync {REPO_ROOT}/wandb/offline-run-*")

: 

## Next step — fine-tune AE/DEC on top of best pretext backbone

Pick the row with the highest `km_gNMI` (or `gmm_gNMI`) and pass its `pretext_backbone.pt` into `02_train_ae.ipynb` by loading the state dict before training:

```python
bb = backbone.MultiModalBackbone(BLOCK_DIMS, PROJ_DIMS, hidden_dim=128, latent_dim=64)
bb.load_state_dict(torch.load(ARTIFACTS / 'models' / '<best-run>' / 'pretext_backbone.pt'))
head = heads.AEHead(bb, BLOCK_DIMS, PROJ_DIMS, hidden_dim=128)
# … then continue with the existing AE training loop
```

Expected payoff per spec §2.1: +5–12% NMI vs cold-start (MVP baseline 0.332).